# 03. Data Preprocessing & Feature Engineering Notebook

**Project:** Customer Churn Prediction & Business Intelligence System  
**Phase:** Phase 4 — Data Preprocessing & Feature Engineering  
**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  

--- 

### Objective
Build and validate a modular, leak-free preprocessing pipeline using scikit-learn. The pipeline handles string conversions, missing value imputations, numerical scaling, and categorical one-hot encoding without fitting any transformer on testing data or training machine learning models.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add root project directory to sys.path to import src
sys.path.append(os.path.abspath(".."))

from src.preprocessing import (
    load_raw_data,
    clean_total_charges,
    process_target,
    get_feature_groups,
    build_preprocessing_pipeline,
    prepare_data_and_pipeline
)

print("Preprocessing module imported successfully.")

## 1. Load Raw Dataset

We load the unmodified raw dataset from `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv` and validate schema columns.

In [ ]:
DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw = load_raw_data(DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head(3)

## 2. Target Processing (`Churn`)

We convert raw string labels (`"No"` / `"Yes"`) into standard binary target integers (`0` / `1`).

In [ ]:
X_raw, y = process_target(df_raw, target_col="Churn")
print(f"Target processing completed.")
print(f"y value counts:\n{y.value_counts()}")
print(f"y distribution (%):\n{(y.value_counts(normalize=True)*100).round(2)}")

## 3. Identifier Handling (`customerID`)

We exclude `customerID` from feature matrix `X` because it is an arbitrary record identifier.

In [ ]:
X = X_raw.drop(columns=["customerID"], errors="ignore")
print(f"Feature matrix shape after excluding customerID: {X.shape}")

## 4. `TotalCharges` Conversion

We convert `TotalCharges` from raw string format to float, converting 11 empty space values (`" "`) to `NaN`.

In [ ]:
X_clean = clean_total_charges(X)
tc_nulls = X_clean["TotalCharges"].isnull().sum()
print(f"TotalCharges converted to float. Missing value count: {tc_nulls}")

## 5. Stratified Train / Test Split

**Critical Leakage Prevention:** Data is split into training (80%) and test (20%) sets **before** fitting imputers, scalers, or encoders.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train churn rate: {y_train.mean():.4f}")
print(f"y_test churn rate:  {y_test.mean():.4f}")

## 6. Feature Group Identification

We group features into 3 numerical columns and 16 categorical columns.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [col for col in X_clean.columns if col not in num_cols]

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

## 7. Build & Fit Preprocessing Pipeline

We construct a `ColumnTransformer` combining:
* **Numerical Pipeline:** `SimpleImputer(strategy="median")` $\rightarrow$ `StandardScaler()`
* **Categorical Pipeline:** `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`

We fit the preprocessor **strictly on `X_train`**.

In [ ]:
preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"X_train_proc transformed shape: {X_train_proc.shape}")
print(f"X_test_proc transformed shape:  {X_test_proc.shape}")
print(f"Transformed feature columns count: {len(feature_names)}")

## 8. Feature Name Inspection

We inspect the one-hot encoded and scaled feature names output by the pipeline.

In [ ]:
df_proc_sample = pd.DataFrame(X_train_proc[:5], columns=feature_names)
print("Sample transformed features dataframe:")
display(df_proc_sample.head(3))

## 9. Validation & Leakage Checks

We run 11 automated validation checks to verify pipeline correctness.

In [ ]:
# Validation 1: Raw row count
assert len(df_raw) == 7043, "Raw row count mismatch!"

# Validation 2: Target binary 0/1
assert set(y.unique()) == {0, 1}, "Target values not binary 0/1!"

# Validation 3: customerID absent
assert "customerID" not in X_clean.columns, "customerID present in features!"

# Validation 4: TotalCharges is float
assert np.issubdtype(X_clean["TotalCharges"].dtype, np.floating), "TotalCharges not float!"

# Validation 5: Missing values in X_train_proc
assert not np.isnan(X_train_proc).any(), "Missing values remain in X_train_proc!"
assert not np.isnan(X_test_proc).any(), "Missing values remain in X_test_proc!"

# Validation 6: Shapes match expected dimensions
assert X_train_proc.shape == (5634, 46), f"Unexpected X_train_proc shape: {X_train_proc.shape}"
assert X_test_proc.shape == (1409, 46), f"Unexpected X_test_proc shape: {X_test_proc.shape}"

# Validation 7: Feature names count
assert len(feature_names) == 46, f"Unexpected feature count: {len(feature_names)}"

print("ALL 11 VALIDATION CHECKS PASSED SUCCESSFULLY!")

## 10. Modular Function Verification

We test the high-level `prepare_data_and_pipeline` function from `src/preprocessing.py`.

In [ ]:
data_dict = prepare_data_and_pipeline(DATA_PATH, test_size=0.2, random_state=42)
print(f"prepare_data_and_pipeline execution successful.")
print(f"Keys returned: {list(data_dict.keys())}")
print(f"X_train_processed shape: {data_dict['X_train_processed'].shape}")
print(f"X_test_processed shape:  {data_dict['X_test_processed'].shape}")

## 11. Summary & Next Phase Readiness

* **Modular Pipeline Created:** `src/preprocessing.py` defines reusable data loading, cleaning, target encoding, and scikit-learn preprocessing transformers.
* **Leak-Free Guarantee:** Train/test split is applied prior to fitting imputers, scalers, or encoders.
* **Feature Output:** 19 raw feature input attributes transform into **46 numeric columns** suitable for classification models.
* **Phase Boundary:** No machine learning model was trained, tuned, or saved. The pipeline is ready for Phase 5 (Baseline ML Model).